# Exploración SECOP II con DuckDB

Consulta el data lake de contratos (Parquet particionado por año) con SQL de DuckDB,
trae el resultado a un `DataFrame` de pandas y lo exporta a Excel para la UAN del
Concejo de Bogotá.

- **Fuente `r2`**: lee el lake en Cloudflare R2 (requiere `.env` con credenciales R2).
- **Fuente `local`**: lee Parquet en disco. Útil sin conexión.

El motor es el mismo que usa la CLI (`secop query` / `secop sql`).

In [ ]:
from secop.query import QueryEngine, export_dataframe
from secop.query.filters import QuerySpec, build_query

# SOURCE = "r2"  # produccion
SOURCE = "local"
# Solo si SOURCE == "local". None usa el lake local por defecto (data/lake/).
LOCAL_PATH = None


def engine() -> QueryEngine:
    return QueryEngine.from_source(
        SOURCE, path=LOCAL_PATH if SOURCE == "local" else None
    )

## 1. Panorama del lake

In [ ]:
with engine() as qe:
    resumen = qe.partition_summary()
    total = qe.count()
    lo, hi = qe.date_range()

print(f"Total contratos: {total:,}")
print(f"fecha_de_inicio_del_contrato: {lo} -> {hi}")
resumen

## 2. SQL directo

La vista se llama `contratos`. `year` es la columna de partición (string).

In [ ]:
with engine() as qe:
    df = qe.run(
        """
        SELECT year,
               count(*) AS contratos,
               sum(TRY_CAST(valor_del_contrato AS DOUBLE)) AS valor_total
        FROM contratos
        WHERE departamento ILIKE '%Bogot%'
        GROUP BY year
        ORDER BY year
        """
    )
df

## 3. Consulta por filtros (equivalente a `secop query`)

Reproduce el caso de `filtro_nit.ipynb`: contratos de la UNP (NIT entidad `900475780`)
desde 2024.

In [ ]:
spec = QuerySpec(
    nit_entidad=["900475780"],
    year_from=2024,
    columns=[
        "id_contrato",
        "nombre_entidad",
        "proveedor_adjudicado",
        "documento_proveedor",
        "valor_del_contrato",
        "fecha_de_inicio_del_contrato",
        "estado_contrato",
    ],
)
sql, params = build_query(spec)
print(sql)

with engine() as qe:
    contratos = qe.run(sql, params)

print(f"{len(contratos):,} filas")
contratos.head(10)

## 4. Análisis rápido en pandas

In [ ]:
contratos["valor_del_contrato"] = contratos["valor_del_contrato"].astype("float64")
(
    contratos.groupby("proveedor_adjudicado")
    .agg(contratos=("id_contrato", "count"), valor_total=("valor_del_contrato", "sum"))
    .sort_values("valor_total", ascending=False)
    .head(15)
)

## 5. Exportar a Excel

Si el resultado supera el límite de filas de Excel, `export_dataframe` avisa y escribe CSV.

In [ ]:
ruta = export_dataframe(contratos, "../data/exports/unp_desde_2024.xlsx")
print("Escrito:", ruta)